# Week 55, Day 3: Primes and modular arithmetic

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w055d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
print(17 % 5, -17 % 5, (3 * 4) % 5, (3 % 5) * (4 % 5) % 5)
print(pow(2, 10, 1000), pow(2, 100, 1000), pow(3, -1, 7))
print((3 * 5) % 7)

In [ ]:
def spf_table(n):
    spf = list(range(n + 1))                 # spf[x] == x means "prime so far"
    p = 2
    while p * p <= n:
        if spf[p] == p:
            for multiple in range(p * p, n + 1, p):
                if spf[multiple] == multiple:
                    spf[multiple] = p
        p += 1
    return spf

def factorise(x, spf):
    factors = {}
    while x > 1:
        p = spf[x]
        factors[p] = factors.get(p, 0) + 1
        x //= p
    return factors

spf = spf_table(1000)
print(factorise(360, spf), factorise(997, spf), factorise(1000, spf))

In [ ]:
def power_mod(a, b, m):
    result, a = 1, a % m
    while b:
        if b & 1:
            result = result * a % m
        a = a * a % m
        b >>= 1
    return result

print(power_mod(2, 10 ** 18, 10 ** 9 + 7), pow(2, 10 ** 18, 10 ** 9 + 7))
print(power_mod(7, 222, 10))           # the last digit of 7 to the power 222

In [ ]:
def extended_gcd(a, b):
    """Return (g, x, y) with a * x + b * y == g == gcd(a, b)."""
    x0, y0, x1, y1 = 1, 0, 0, 1
    while b:
        q = a // b
        a, b = b, a - q * b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return a, x0, y0

g, x, y = extended_gcd(240, 46)
print(g, x, y, 240 * x + 46 * y)
g, x, _ = extended_gcd(3, 7)
print(x % 7, pow(3, -1, 7))

In [ ]:
def crt(r1, m1, r2, m2):
    k = (r2 - r1) * pow(m1, -1, m2) % m2
    return (r1 + m1 * k) % (m1 * m2), m1 * m2

x, m = crt(2, 3, 3, 5)
x, m = crt(x, m, 2, 7)
print(x, m, [x % 3, x % 5, x % 7])

> **Key idea.** A smallest-prime-factor sieve factorises any number below its limit in O(log x) steps; trial division to √x handles single numbers up to about 10¹². Mod m, addition, subtraction and multiplication respect remainders, so reduce at every step. Fast exponentiation squares and multiplies by the bits of the exponent: pow(a, b, m). An inverse of a mod m exists when gcd(a, m) = 1 and comes from the extended Euclidean algorithm (or pow(a, −1, m)); mod a prime p it is also a^(p − 2). The Chinese remainder theorem combines congruences with coprime moduli, and Euler's totient φ(n) gives a^φ(n) ≡ 1 (mod n).

### Exercise: Factorising fast

Write `spf_table(n)` (the smallest prime factor of every number 0 to n, with `spf[0]` and `spf[1]` left as 0 and 1), `factorise(x, spf)` (a dict from prime to exponent, `{}` for 1), and `divisor_count(x, spf)`: the number of divisors of x, which is the product of (exponent + 1) over its primes.

In [ ]:
def spf_table(n):
    ___

def factorise(x, spf):
    ___

def divisor_count(x, spf):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef test_factors():\n    \"sieves\"\n    spf = spf_table(1000)\n    assert spf[:12] == [0, 1, 2, 3, 2, 5, 2, 7, 2, 3, 2, 11] and spf[97] == 97 and spf[91] == 7\n    assert factorise(360, spf) == {2: 3, 3: 2, 5: 1} and factorise(1, spf) == {} and factorise(97, spf) == {97: 1}\n    assert divisor_count(60, spf) == 12 and divisor_count(1, spf) == 1\n    big = spf_table(200_000)\n    rng = random.Random(1)\n    for _ in range(300):\n        x = rng.randint(1, 200_000)\n        f = factorise(x, big)\n        assert math.prod(p ** e for p, e in f.items()) == x and all(big[p] == p for p in f)\n        if x < 3000:\n            assert divisor_count(x, big) == sum(1 for d in range(1, x + 1) if x % d == 0)")

### Exercise: Square and multiply

Write `power_mod(a, b, m)` by repeated squaring, without `pow` or `**` (b ≥ 0, m ≥ 1). Then `last_digits(a, b, k)`: the last k digits of a^b as a string with leading zeros (so the last 3 digits of 2^10 are `"024"`).

In [ ]:
def power_mod(a, b, m):
    ___

def last_digits(a, b, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_powers():\n    \"fast powers\"\n    assert power_mod(2, 10, 1000) == 24 and power_mod(7, 0, 13) == 1 and power_mod(5, 3, 1) == 0\n    assert last_digits(2, 10, 3) == \"024\" and last_digits(7, 222, 1) == \"9\" and last_digits(3, 10 ** 15, 5) == f\"{pow(3, 10 ** 15, 10 ** 5):05d}\"\n    rng = random.Random(2)\n    for _ in range(300):\n        a, b, m = rng.randint(0, 10 ** 9), rng.randint(0, 10 ** 18), rng.randint(1, 10 ** 9)\n        assert power_mod(a, b, m) == pow(a, b, m)\n    assert \"pow(\" not in source and \"**\" not in source")

### Exercise: Dividing on a clock

Write `extended_gcd(a, b)` returning `(g, x, y)` with `a * x + b * y == g`, then `mod_inverse(a, m)`: the inverse of a mod m between 0 and m − 1, raising `ValueError` when it doesn't exist (don't use `pow`). Then `mod_divide(a, b, m)`: a "divided by" b mod m.

In [ ]:
def extended_gcd(a, b):
    ___

def mod_inverse(a, m):
    ___

def mod_divide(a, b, m):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef test_inverse():\n    \"inverses\"\n    g, x, y = extended_gcd(240, 46)\n    assert g == 2 and 240 * x + 46 * y == 2\n    assert mod_inverse(3, 7) == 5 and mod_inverse(10, 17) == 12 and mod_divide(4, 3, 7) == 6\n    for bad in ((2, 4), (6, 9)):\n        try:\n            mod_inverse(*bad)\n            assert False\n        except ValueError:\n            pass\n    rng = random.Random(3)\n    for _ in range(300):\n        a, b = rng.randint(0, 10 ** 12), rng.randint(1, 10 ** 12)\n        g, x, y = extended_gcd(a, b)\n        assert g == math.gcd(a, b) and a * x + b * y == g\n        m = rng.randint(2, 10 ** 9)\n        if math.gcd(a, m) == 1:\n            inv = mod_inverse(a, m)\n            assert 0 <= inv < m and a * inv % m == 1\n    assert \"pow(\" not in source")

### Exercise: Remainders together

Write `crt(remainders, moduli)`: `(x, M)`, the smallest non-negative x with `x % m == r` for every pair, and M the product of the moduli, combining two at a time. The moduli must be pairwise coprime: raise `ValueError` otherwise. (You may use `pow(a, -1, m)` here.)

In [ ]:
def crt(remainders, moduli):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef test_crt():\n    \"Chinese remainders\"\n    assert crt([2, 3, 2], [3, 5, 7]) == (23, 105) and crt([0], [7]) == (0, 7) and crt([], []) == (0, 1)\n    assert crt([1, 2, 3, 4], [5, 7, 9, 11])[0] % 9 == 3\n    try:\n        crt([1, 2], [4, 6])\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(4)\n    primes = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31]\n    for _ in range(200):\n        ms = rng.sample(primes, rng.randint(1, 5))\n        x = rng.randrange(math.prod(ms))\n        got = crt([x % m for m in ms], ms)\n        assert got == (x, math.prod(ms))")

### Exercise: Euler's totient

Write `phi(n)`: Euler's totient of n ≥ 1, by trial division to find its distinct primes and the product formula (whole-number arithmetic only: `result -= result // p`). Then `phi_table(n)`: φ of every number 0 to n (with φ(0) = 0), by a sieve: start with `t[i] = i`, and for each prime p (where `t[p]` is still p), apply `t[k] -= t[k] // p` to every multiple k of p.

In [ ]:
def phi(n):
    ___

def phi_table(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\ndef test_phi():\n    \"totients\"\n    assert [phi(n) for n in range(1, 13)] == [1, 1, 2, 2, 4, 2, 6, 4, 6, 4, 10, 4]\n    assert phi(10 ** 9 + 7) == 10 ** 9 + 6 and phi(2 ** 40) == 2 ** 39\n    table = phi_table(2000)\n    assert table[:6] == [0, 1, 1, 2, 2, 4]\n    for n in range(1, 2001):\n        assert table[n] == phi(n)\n    for n in range(1, 200):\n        assert phi(n) == sum(1 for k in range(1, n + 1) if math.gcd(k, n) == 1)\n        for a in (2, 3, 7):\n            if math.gcd(a, n) == 1:\n                assert pow(a, phi(n), n) == 1 % n")

### Bug hunt: Squares slip through

Trial division must try every d with `d * d <= n`. This stops at `d * d < n`, so it never tries 2 for 4 or 7 for 49: every square of a prime slips through, and it prints `[4, 9, 25, 49]`. Fix the condition so it prints `[]`.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    d = 2
    while d * d < n:
        if n % d == 0:
            return False
        d += 1
    return True

print([n for n in (4, 9, 25, 49) if is_prime(n)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"up to the square root\"\n    expect_output(\"[]\")")

### Bug hunt: Fermat's inverse on a non-prime

`pow(a, m - 2, m)` is the inverse of a only when m is **prime**. For m = 8 it gives the wrong answer, and this prints `False`. Use `pow(a, -1, m)`, which works for any m with gcd(a, m) = 1, so it prints `True`.

In [ ]:
a, m = 3, 8
inv = pow(a, m - 2, m)
print(a * inv % m == 1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"general inverses\"\n    expect_output(\"True\")")

### Bug hunt: Squaring the result

In square and multiply, it's the **base** that is squared each step, not the result. This squares `result` instead and prints `609` for 3¹³ mod 1000. Square `a` so it prints `323`.

In [ ]:
def power_mod(a, b, m):
    result = 1
    while b:
        if b & 1:
            result = result * a % m
        result = result * result % m
        b >>= 1
    return result

print(power_mod(3, 13, 1000))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"square the base\"\n    expect_output(\"323\")")

### Puzzle: Extended Euclid

Arrange the lines into the extended Euclidean algorithm. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # -7 % 3
q = ___   # pow(2, 20, 1000)
r = ___   # the inverse of 4 mod 9
s = ___   # phi(36)
t = ___   # the smallest x with x % 4 == 3 and x % 5 == 2

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 576 and r == 7, \"p, q and r\"\n    assert s == 12 and t == 7, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a toy RSA

In 1977 Ron Rivest, Adi Shamir and Leonard Adleman published a public-key system: anyone can **encrypt** with a public key, but only the holder of the private key can **decrypt**. (Clifford Cocks had found the same idea in 1973 at GCHQ, kept secret until 1997.) Build a toy version. It's for understanding only: real RSA uses primes hundreds of digits long and careful padding, and you should always use a tested library.

- `make_keys(p, q, e=65537)`: from two distinct primes, n = p × q and φ = (p − 1)(q − 1); e must be coprime to φ (raise `ValueError` otherwise); d is the inverse of e mod φ. Return `((n, e), (n, d))`, the public and private keys.
- `encrypt(m, public)` = m^e mod n and `decrypt(c, private)` = c^d mod n, for whole numbers 0 ≤ m < n (`ValueError` otherwise).
- `encrypt_text(text, public)`: turn the text into bytes (UTF-8), the bytes into one big number with `int.from_bytes(data, "big")`, check it's smaller than n, and encrypt; `decrypt_text(c, private)` reverses it (find the byte length from the number's `bit_length`).
- `sign(m, private)` = m^d mod n, and `verify(m, signature, public)`: True if signature^e mod n equals m.
- `crack(public)`: recover the private key by factorising n with trial division, which is only possible because the toy n is small. That is exactly why real n are enormous.

In [ ]:
import math

def make_keys(p, q, e=65537):
    ___

def encrypt(m, public):
    ___

def decrypt(c, private):
    ___

def encrypt_text(text, public):
    ___

def decrypt_text(c, private):
    ___

def sign(m, private):
    ___

def verify(m, signature, public):
    ___

def crack(public):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_rsa():\n    \"a toy RSA\"\n    public, private = make_keys(61, 53, 17)\n    assert public == (3233, 17) and private == (3233, 2753)\n    assert encrypt(65, public) == 2790 and decrypt(2790, private) == 65\n    try:\n        make_keys(61, 53, 6)\n        assert False\n    except ValueError:\n        pass\n    try:\n        encrypt(4000, public)\n        assert False\n    except ValueError:\n        pass\n    p, q = 1_000_003, 999_983\n    public, private = make_keys(p, q)\n    for m in (0, 1, 42, 123456789, p * q - 1):\n        assert decrypt(encrypt(m, public), private) == m\n    pub2, priv2 = make_keys(2_147_483_647, 2_305_843_009_213_693_951)\n    c = encrypt_text(\"hi, Atlas\", pub2)\n    assert decrypt_text(c, priv2) == \"hi, Atlas\"\n    assert verify(99, sign(99, private), public) and not verify(98, sign(99, private), public)\n    assert crack(public) == private and crack(make_keys(61, 53, 17)[0]) == (3233, 2753)")

## Warm-up

Python's `%` always gives a result between 0 and the modulus minus 1, even for negative numbers. Remainders can be taken at any step of a product without changing the final remainder. `pow(a, b, m)` computes a^b mod m without ever building a^b, and `pow(3, -1, 7)` finds the number that multiplies by 3 to give 1 on a clock of 7.

## Sieves that remember

Week 14's sieve marked composites. A small change makes it far more useful: instead of a True/False flag, store each number's **smallest prime factor** (spf). When p is prime, mark every multiple from p² upwards whose entry is still empty with p.

Then any number x below the limit factorises in O(log x) steps: divide by `spf[x]`, then by `spf[x // spf[x]]`, and so on, until 1. That's how you factorise millions of numbers quickly (counting divisors of everything below a limit, for example). Building the table takes about O(n log log n) time, like the ordinary sieve.

For a single large number, **trial division** checks divisors up to √x: if x has no factor that small, it's prime (a composite has a factor no bigger than its square root). That's about √x steps: fine for x up to 10¹², slow for 10¹⁸. Friday's Miller-Rabin test checks primality of huge numbers fast.

## Clock arithmetic

Arithmetic **mod m** keeps only remainders: 0 to m − 1, wrapping round like a clock. Addition, subtraction and multiplication all **respect** remainders:

- (a + b) mod m = ((a mod m) + (b mod m)) mod m, and the same for − and ×.

So you can reduce after every step, and numbers never grow beyond m². That's why programming contests ask for answers "mod 1,000,000,007" (a prime just above 10⁹), and why hash functions, checksums and random number generators work mod 2³² or 2⁶⁴. Python's integers never overflow, but reducing still keeps them small and fast.

Two things don't carry over directly: **division** (you need an inverse, below) and comparing sizes (7 mod 5 is less than 4 mod 5, though 7 > 4).

## Fast powers

a^b mod m by repeated squaring (Week 16's fast powers, now mod m): write b in binary. a¹, a², a⁴, a⁸, ... are each the square of the last; multiply together the ones whose bit is set in b. That's O(log b) multiplications of numbers below m, so 2^(10¹⁸) mod m takes about 60 steps. Python's built-in `pow(a, b, m)` does exactly this, in C.

**Fermat's little theorem** (Pierre de Fermat, 1640): if p is prime and a isn't a multiple of p, then a^(p − 1) mod p = 1. It's the basis of Friday's primality tests, and it gives one inverse: a^(p − 2) times a is a^(p − 1), which is 1, so a^(p − 2) is the inverse of a mod a **prime** p.

## Inverses

To divide by a mod m, multiply by its **inverse**: the x with a × x mod m = 1. It exists exactly when gcd(a, m) = 1 (a and m share no factor). Find it with the **extended Euclidean algorithm**, which returns g = gcd(a, b) together with whole numbers x and y such that a × x + b × y = g (Bézout's identity). If g is 1, then a × x mod b is 1, so x mod b is the inverse.

The extension tracks how each remainder is made from a and b as Euclid's algorithm runs. Since Python 3.8, `pow(a, -1, m)` computes the inverse directly (and raises `ValueError` if it doesn't exist).

## Combining clocks

A number leaves remainder 2 when divided by 3, remainder 3 by 5, and remainder 2 by 7. What is it? The **Chinese remainder theorem** (in Sunzi's *Mathematical Classic*, around the 3rd to 5th century) says: if the moduli are pairwise coprime, there's exactly one answer modulo their product. Combine two at a time: from x ≡ r₁ (mod m₁) and x ≡ r₂ (mod m₂), write x = r₁ + m₁ × k, and solve m₁ × k ≡ r₂ − r₁ (mod m₂) using the inverse of m₁ mod m₂.

**Euler's totient** φ(n) counts the numbers from 1 to n coprime to n. From the factorisation, φ(n) = n × the product of (1 − 1/p) over its distinct primes. Euler's theorem generalises Fermat's: a^φ(n) mod n = 1 when gcd(a, n) = 1. That's the identity RSA rests on (today's challenge).

## Practice

## Quiz

## Daily challenge

## Before you go

Find a number that leaves remainder 1 when divided by 2, 3, 4, 5 and 6, but is divisible by 7. (It's an old puzzle; try the Chinese remainder idea.) Tomorrow: counting and combinatorics.